# <h1 style='color:#1B3A6B;'> INFORME DE SOLUCION </h1>

----

# <span style='color:#2E86AB;'> Proyecto: Predicción de Cancelación de Clientes (Churn) para Interconnect </span>

---
---

# <span id="objetivo" style='color:#1B3A6B;'> 1. OBJETIVO </span>

---
   * Desarrollar un modelo de clasificación que prediga si un cliente cancelará su servicio e identificar anticipadamente a los clientes en riesgo de abandono.


---
# <span style='color:#1B3A6B;'> 2. RESULTADO FINAL </span>
---

## <span style='color:#2E86AB;'> 2.1. PASOS REALIZADOS Y OMITIDOS </span>

### <span style='color:#2E86AB;'> Pasos Realizados </span>

**1. Exploración EDA**

   * Explorar y comprender los datos de los 4 archivos (`contract`, `personal`, `internet`, `phone`) identificando tipos de variables, valores ausentes, duplicados y la relación entre ellos a través de la columna `customerid`.
   
**2. Procesamiento**

   * Unificar las fuentes de datos en un único dataset consolidado, tratando adecuadamente a los clientes que no tienen ciertos servicios como `phone` y `internet`, obteniendo un total de 7043 filas y 20 columnas.

   * Construir la variable objetivo (`target`) a través de `enddate`, obteniendo (Activos(0): 5174 (73.5 %)), (Cancelados (1): 1869 (26.5 %)), analizando el balance de las 2 clases.

   * Realizar ingeniería de características, incluyendo la antiguedad de los clientes usando como fecha de referencia 01 de febrero de 2020, tipo de contrato, método de pago, servicios contratados, entre otros.

   * Tratamiento de nulos, en la columna `totalcharges` los valores ausentes fueron remplazados con '0', mientras que los valores nulos en los datasets de clientes de las tablas (`Phone` y `Internet`) son remplazados con el argumento de 'Sin servicio', evitando eliminar o ignorar algunos de estos valores ausentes.
  
   * La codificación de las variables categóricas se realizo de dos maneras: a) Variables binarias (Si/No) en un mapeo directo a `0/1`; b) Variables categóricas con más de 2 variables, se manejó con `One-Hot Encoding` ya que no tienen un orden natural, obteniendo como resultado un total de 33 columnas incluidos `target` conservando las mismas filas (7043). 
  
**3. Análisis Exploratorio y de Correlaciones**

   * Las columnas `begindate`, `enddate` y `customerid` son excluidas de cualquier análisis o modelos, ya que `enddate` es la fuente directa del target y usarla causaría una fuga de información y `customerid` no aporta ninguna información predictiva real, por lo tanto, las columnas disponibles para el análisis y modelo son de `29 columnas + columna 'target' (objetivo)`.

   * En la distribución de la variable objetivo `'target'`(churn), se observó un desbalance de clases: clientes activos (5174 = 73.5%) y clientes cancelados (1869 = 26.5%).
  
   * En las figuras 2 (Variables numéricas vs. churn (boxplots)) y 4 (Matriz de correlación), se observa una similitud en los resultados: indicando que `antiguedad_meses` a mayor antigüedad de los clientes, menor es la probabilidad de cancelar; de igual manera se puede evidenciar en la figura 8 en dónde, `antiguedad_meses (44.48 %)` es por lejos la variable más importante. Esto confirma lo observado en el EDA (sección 4.2): los clientes con menor tiempo en Interconnect son los que más probabilidad tienen de cancelar, mientras que la lealtad se consolida con el tiempo.

**4. Preparación para el Modelo**
  
   * Se procede a dividir el conjunto de datos usando `stratify= y`, con el objetivo de conservar la misma proporción de clases (churn/no churn) en ambos conjuntos de datos, debido al desbalance detectado en el EDA y como resultado se obtiene: `train` (5282 filas) - `test` (1761 filas) y una proporción de churn en train - test de `0.265`.

   *  El escalado es necesario para los modelos de Regresión Logística (sensible a escala), sin embargo no es estrictamente necesario para modelos basados en árboles (Random Forest, LightGBM, CatBoost), pero no les afecta negativamente, así que preparamos una versión escalada para reutilizar en todos los modelos lineales.


**5. Entrenar y comparar Modelos de Regresión**

   * Comparamos 4 modelos base (`Regresión Logística`, `Random Forest`, `LightGBM/CatBoost`) usando validación cruzada estratificada con 5 folds sobre el conjunto de entrenamiento, optimizando el AUC-ROC (métrica principal) y manejando el desbalance de las clases.
  
   * Seleccionamos los 2 mejores modelos bases y afinamos los hiperparámetros de cada uno, incorporando el desbalance de clases detectadas en el EDA (26.5% churn) obteniendo: 

|Modelos|AUC-ROC promedio|
|---|---|
|**CatBoost**|**0.8633**|
|**LightGBM**|**0.8605**|
|Logistic Regression|0.8471|
|Random Forest|0.8275|

   * El modelo con mejor AUC-ROC en la comparación con hiperparámetros ajustados fue `CatBoost`, antes de entrenarlo se calcula `scale_pos_weight = 2.767 ` para compensar el desbalance de clases, de modo que el modelo penalice más a los errores sobre la clase minoritaria durante el entrenamiento.

**6. Evaluación Final**

   * El modelo final `CatBoost` alcanzó un `AUC-ROC de 0.8673 (86.73%)`, superando el umbral mínimo de 0.75 requerido por el proyecto.
   
   * La `exactitud (accuracy)` por sí sola no es suficiente para evaluar este problema dado el desbalance de clases; por eso el AUC-ROC es la métrica principal de referencia.

---   
### <span style='color:#2E86AB;'> Pasos Omitidos </span>


Durante las etapas de evaluación, análisis, procesamiento, entrenamiento de modelos, ajuste de hiperparámetros y evaluación final del conjunto de datos, se ejecutaron de manera rigurosa todos los procedimientos planificados, garantizando así el cumplimiento del 100% del plan de trabajo.

---
## <span style='color:#2E86AB;'> 2.2. DIFICULTADES ENCONTRADAS </span>
---


**1. Fuga de datos**

 * Para evitar la fuga de datos al momento de entrenar los modelos, guardamos las columnas derivadas de `enddate` para poder `EXCLUIRLAS` del análisis de churn, además las columnas como: (`begindate` y `enddate`) no se usaron como `features` del modelo directamente en su forma original (fecha), pero sí podemos extraer de ellas una feature útil y no contaminada, la antigüedad del cliente en meses (`'antiguedad_meses'`), calculada respecto a la fecha de referencia del proyecto (1 de febrero de 2020) para los clientes que siguen activos.

**2. Valores Nulos**

   * La columna `totalcharges` se encontraba como texto y algunos valores como espacios en blanco, hubo inconvenientes durante las gráficas de `boxplots` mostrando gráficas ilegibles, anteriormente en el punto de tratamiento de nulos no se identificó estos valores, dando inconsistencia al momento de realizar la figura 2 (Variables numéricas vs. churn), esto ocurre en clientes muy nuevos con 0 meses de antigüedad o aún sin facturación total.

     La solución que se implemento es convertir la columna `'totalcharges'` a un tipo de datos numéricos ('float' para este caso); complementando con `errors='coerce'`, tratando a los valores que no se puedan transformar en número como: un espacio en blanco ' ', un texto vacío '', o un error como 'N/A', y que no lance un error; en su lugar, lo convierta en un valor nulo (NaN), resolviendo de forma segura los posibles errores de texto o espacios en blanco, para luego dichos valores nulos ser reemplazados por el valor 0.

**3. Desbalance de Clases**

   * El conjunto de datos presenta una distribución desbalanceada, donde los clientes activos representan el 73.5% (5,174 casos) y los clientes cancelados el 26.5% (1,869 casos). Para abordar esta desigualdad sin sesgar el modelo, se aplicó una estrategia en dos partes:
     
     - **Preservación de la estructura de datos:**
    
       Se utilizó el parámetro `stratify= y` en la partición inicial y una validación cruzada mediante `StratifiedKFold(n_splits=5, shuffle=True)`. Esto garantiza que tanto los conjuntos de entrenamiento/prueba como cada uno de los `5 folds` mantengan exactamente la misma proporción original de la variable objetivo (churn / no churn).
      
      - **Ajuste en el algoritmo (CatBoost):**
    
        Durante el entrenamiento, se incorporó el parámetro `scale_pos_weight`, configurado con la relación de la clase mayoritaria sobre la minoritaria (negativos / positivos). Esto `incrementa la penalización` de los errores sobre los clientes que cancelan es decir la clase minoritaria, obligando al modelo a priorizar su correcta detección.

---
## <span style='color:#2E86AB;'> 2.3. PASOS CLAVE PARA RESOLVER LA TAREA </span>
---

Las decisiones determinantes para alcanzar el objetivo del proyecto fueron tres:

   1. Comparar varios candidatos antes de ajustar hiperparámetros, en vez de fijarse solo en uno. Tras la validación cruzada inicial con los 4 modelos base, `CatBoost`y `LightGBM` quedaron con un `AUC-ROC` muy cercano entre sí, una diferencia dentro del margen de su propia desviación estándar, por lo que se ajustaron hiperparámetros a ambos en lugar de descartar a 'LightGBM' solo por haber quedado en segundo lugar por un margen tan pequeño. 
  
   2. Controlar el `desbalance` de clases mediante `scale_pos_weight` en el modelo final, calculado como la razón entre clientes activos y cancelados `(73.5%/26.5%)`, para que el modelo penalice más los errores sobre la clase minoritaria (churn) durante el entrenamiento.

   3. Evitar la fuga de información, excluyendo del conjunto de `features` las columnas derivadas directamente de la columna `enddate` (la propia fuente del target), que de haberse incluido habrían inflado artificialmente el desempeño del modelo sin representar información realmente disponible al momento de predecir.

---
## <span style='color:#2E86AB'> 2.4 MODELO FINAL Y NIVEL DE CALIDAD </span>
---
   * El `AUC-ROC` obtenido con el modelo `CatBoost` es de `86.7%` (métrica principal) clasificada como `'muy bueno'` en la escala estándar de evaluación de modelos, con una `exactitud (accuracy)` de `77.80%` (métrica secundaria) y con una puntuación estimada del proyecto `SP= 5`.

   * Respaldando el valor AUC-ROC se observa en la figura 6, la curva muestra un `Excelente Rendimiento (86.7%)` acercándose a la esquina superior izquierda, indicando que el modelo tiene una alta capacidad de distinguir entre clientes que cancelan y los activos.
  
   * En la Figura 7, la matriz de confusión demuestra que el modelo logra una `sensibilidad (recall) del 80%`, identificando correctamente a `8 de cada 10 clientes` que cancelan el servicio. Por otro lado, la `precisión` se sitúa en un `55.6% (374/672)`. Este comportamiento, con un `recall alto` combinado con una `precisión moderada`, es plenamente coherente con el `AUC-ROC de ~0.86-0.87` reportado, un rendimiento robusto para un escenario con desbalance moderado de clases.

---
# <span style='color:#1B3A6B;'>3. RECOMENDACIONES DE NEGOCIO </span>
---

* Priorizar la retención en los primeros meses de contrato
`'antiguedad_meses'` es, por lejos, la variable más importante para predecir cancelación (44.48%). El riesgo de churn se concentra en clientes nuevos. Se recomienda activar una campaña de bienvenida o seguimiento proactivo durante los primeros 3-6 meses, por ejemplo: contacto de soporte, encuesta de satisfacción temprana; con el fin de reforzar la relación antes de que el cliente considere cancelar.

* Incentivar la migración de contratos mes a mes hacia contratos anuales, los clientes con contrato `'month-to-month'` (figura 3) presentan la tasa de churn más alta, mientras que los contratos de 1-2 años (`type_two year`, `type_one year`) son predictores relevantes de permanencia. Se recomienda ofrecer descuentos o beneficios exclusivos como: meses gratis, prioridad en soporte técnicos; sobre todo dirigido a los clientes con contrato mensual que llevan más meses activos, incentivando así la migración a un contrato de mayor duración.

   
